# Natural-Language Play Entry for a Flag Football Stat Tracker: Final Evaluation
COMP 365 Individual Project, final submission notebook.

**Question:** Can an LLM turn a plain-English description of a flag football play into a structured stat record more accurately than a rule-based baseline, and how much do few-shot examples and model size matter?

**Systems compared (same 110 held-out sentences, same scoring):**
1. Rule-based baseline (regex + keywords, frozen before this dataset was written; no AI)
2. Claude Haiku 4.5, zero-shot (schema + rules only)
3. Claude Haiku 4.5, few-shot (same prompt + 20 dev examples)
4. Claude Sonnet 5.5, few-shot (same prompt as 3, larger model)

**Data:** `play_dataset_v1.csv` (130 sentences, embedded below): 20 dev (prompt examples only), 80 test, 30 hard. Sentences and primary labels were written by Claude (AI assistance) following `LABELING_GUIDE.md`; a second pass over 20 random rows (seed 365) was also done by Claude. See the paper for limits.

**To run:** Runtime > Run all. When asked, paste your Anthropic API key (hidden input; never saved). The baseline needs no key. Outputs: `results.json`, `results_table.csv`, `predictions.json`, `errors.csv`, `fig_main.png`, `fig_category.png`.

In [ ]:
# 1. Setup
%pip install -q --upgrade anthropic pandas matplotlib
import os, re, json, csv, io, math, getpass, sys, time, platform, datetime
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import matplotlib.pyplot as plt

HAIKU  = "claude-haiku-4-5-20251001"
SONNET = "claude-sonnet-5-5"
REPEATS = 2        # repeated runs per LLM system (default temperature, so outputs can vary)
WORKERS = 6        # parallel API calls
print("python", platform.python_version(), "|", datetime.date.today())

In [ ]:
DATA_CSV = """id,sentence,play_type,passer,receiver,runner,yards,result,split,difficulty
d01,Marcus threw to Jaylen for 12 yards,pass,Marcus,Jaylen,,12,complete,dev,easy
d02,Devon ran for 8 yards,run,,,Devon,8,none,dev,easy
d03,Marcus pass to Tasha incomplete,pass,Marcus,Tasha,,0,incomplete,dev,easy
d04,Riley rushed for a touchdown from 15 yards out,run,,,Riley,15,touchdown,dev,easy
d05,Marc found Dev deep for 25 yards,pass,Marcus,Devon,,25,complete,dev,easy
d06,"Sam handed off to Jaylen, no gain",run,,,Jaylen,0,none,dev,easy
d07,"Marcus was picked off, intended for Riley",pass,Marcus,Riley,,0,interception,dev,easy
d08,Tasha caught a 10 yard pass from Marcus,pass,Marcus,Tasha,,10,complete,dev,easy
d09,"Flag on the defense, 5 yards",penalty,,,,5,none,dev,easy
d10,J-Money took it 6 yards,run,,,Jaylen,6,none,dev,easy
d11,"Riley caught a quick screen from Sam, 4 yards",pass,Sam,Riley,,4,complete,dev,easy
d12,Tasha ran it up the middle for 7,run,,,Tasha,7,none,dev,easy
d13,Sam's throw to Dev fell incomplete,pass,Sam,Devon,,0,incomplete,dev,easy
d14,"Offensive holding, 10 yards",penalty,,,,-10,none,dev,easy
d15,"Marcus hit Jay in stride for a touchdown, 35 yards",pass,Marcus,Jaylen,,35,touchdown,dev,easy
d16,Devon got stopped for a loss of 2,run,,,Devon,-2,none,dev,easy
d17,"Sam was intercepted, he was looking for Tasha",pass,Sam,Tasha,,0,interception,dev,easy
d18,Jaylen scored on a 9 yard run,run,,,Jaylen,9,touchdown,dev,easy
d19,"Delay of game, 5 yards",penalty,,,,-5,none,dev,easy
d20,"Marcus to Riley, 11 yards",pass,Marcus,Riley,,11,complete,dev,easy
t01,Marcus completed a pass to Devon for 14 yards,pass,Marcus,Devon,,14,complete,test,standard
t02,Sam found Tasha over the middle for 9,pass,Sam,Tasha,,9,complete,test,standard
t03,"Riley made the catch, 6 yards, from Marcus",pass,Marcus,Riley,,6,complete,test,standard
t04,"Marcus lofted one to Jay down the sideline, 27 yards",pass,Marcus,Jaylen,,27,complete,test,standard
t05,Tash caught Sam's throw for 12,pass,Sam,Tasha,,12,complete,test,standard
t06,Sam to Dev for 5 yards,pass,Sam,Devon,,5,complete,test,standard
t07,"Marcus zipped it to Riley, gain of 8",pass,Marcus,Riley,,8,complete,test,standard
t08,Jaylen pulled in a 16 yard pass from Sam,pass,Sam,Jaylen,,16,complete,test,standard
t09,Marc connected with Tasha on a 22 yard throw,pass,Marcus,Tasha,,22,complete,test,standard
t10,Devon got a short pass from Marcus and picked up 3,pass,Marcus,Devon,,3,complete,test,standard
t11,"#9 hauled it in for 11 yards, thrown by #15",pass,Sam,Tasha,,11,complete,test,standard
t12,"Marcus threw a bullet to #4, 19 yards",pass,Marcus,Jaylen,,19,complete,test,standard
t13,Sam dropped it off to Riley for 2,pass,Sam,Riley,,2,complete,test,standard
t14,Riley was wide open and Marcus hit her for 13,pass,Marcus,Riley,,13,complete,test,standard
t15,Tasha got the screen pass from Sam and went 15 yards,pass,Sam,Tasha,,15,complete,test,standard
t16,"Jay made a nice grab from Marcus, 10 yards",pass,Marcus,Jaylen,,10,complete,test,standard
t17,"Dev caught it, 7 yards, Sam threw it",pass,Sam,Devon,,7,complete,test,standard
t18,"Marcus fired a pass to Tasha, 18",pass,Marcus,Tasha,,18,complete,test,standard
t19,Sammy to Jaylen for 20 yards,pass,Sam,Jaylen,,20,complete,test,standard
t20,Marcus threw it short to Devon and he gained 4,pass,Marcus,Devon,,4,complete,test,standard
t21,"Sam threw a touchdown pass to Tasha, 24 yards",pass,Sam,Tasha,,24,touchdown,test,standard
t22,"Marcus to Jay in the end zone for six, 12 yards",pass,Marcus,Jaylen,,12,touchdown,test,standard
t23,TD! Sam hit Devon from 8 yards out,pass,Sam,Devon,,8,touchdown,test,standard
t24,"Riley caught a touchdown from Marcus, 30 yards",pass,Marcus,Riley,,30,touchdown,test,standard
t25,Marcus's pass for Tasha was incomplete,pass,Marcus,Tasha,,0,incomplete,test,standard
t26,Sam threw it out of bounds,pass,Sam,,,0,incomplete,test,standard
t27,"Marcus missed Devon, incomplete",pass,Marcus,Devon,,0,incomplete,test,standard
t28,Riley dropped Sam's pass,pass,Sam,Riley,,0,incomplete,test,standard
t29,"Incomplete, Marcus to Jaylen",pass,Marcus,Jaylen,,0,incomplete,test,standard
t30,Sam's pass to Dev was batted down,pass,Sam,Devon,,0,incomplete,test,standard
t31,Marcus threw it away under pressure,pass,Marcus,,,0,incomplete,test,standard
t32,Jay couldn't hold on to the pass from Marcus,pass,Marcus,Jaylen,,0,incomplete,test,standard
t33,"Marcus threw an interception, targeting Jaylen",pass,Marcus,Jaylen,,0,interception,test,standard
t34,"Sam's pass was picked off, intended for Riley",pass,Sam,Riley,,0,interception,test,standard
t35,Interception! Marcus was looking for Tasha,pass,Marcus,Tasha,,0,interception,test,standard
t36,The defense intercepted Sam's throw to Devon,pass,Sam,Devon,,0,interception,test,standard
t37,Marcus got picked off,pass,Marcus,,,0,interception,test,standard
t38,Jaylen ran for 6 yards,run,,,Jaylen,6,none,test,standard
t39,Devon took the handoff and gained 9,run,,,Devon,9,none,test,standard
t40,Tasha rushed up the middle for 4,run,,,Tasha,4,none,test,standard
t41,Marcus scrambled for 8 yards,run,,,Marcus,8,none,test,standard
t42,Riley got 3 yards on the ground,run,,,Riley,3,none,test,standard
t43,Sam kept it and ran for 12,run,,,Sam,12,none,test,standard
t44,Dev ran for a touchdown from 20 yards,run,,,Devon,20,touchdown,test,standard
t45,Jaylen scored on a 14 yard run,run,,,Jaylen,14,touchdown,test,standard
t46,Tasha got stopped for no gain,run,,,Tasha,0,none,test,standard
t47,Marcus ran for a loss of 4,run,,,Marcus,-4,none,test,standard
t48,Devon was dropped for a 2 yard loss,run,,,Devon,-2,none,test,standard
t49,Riley took it 10 yards,run,,,Riley,10,none,test,standard
t50,#12 ran 5 yards,run,,,Devon,5,none,test,standard
t51,J-Money rushed for 7,run,,,Jaylen,7,none,test,standard
t52,Sam handed it to Tasha and she ran for 11,run,,,Tasha,11,none,test,standard
t53,Marcus ran it in from 6 yards for a score,run,,,Marcus,6,touchdown,test,standard
t54,"Offense jumped early, false start, 5 yards",penalty,,,,-5,none,test,standard
t55,"Defense was offside, 5 yards",penalty,,,,5,none,test,standard
t56,"Pass interference on the defense, 10 yards",penalty,,,,10,none,test,standard
t57,"Illegal motion on the offense, 5 yards",penalty,,,,-5,none,test,standard
t58,"Flag for holding on the offense, 10 yards",penalty,,,,-10,none,test,standard
t59,"Illegal contact on the defense, 5 yards",penalty,,,,5,none,test,standard
t60,"Delay of game on the offense, 5 yards",penalty,,,,-5,none,test,standard
t61,"Encroachment on the defense, 5 yards",penalty,,,,5,none,test,standard
t62,"Offensive pass interference, 10 yards",penalty,,,,-10,none,test,standard
t63,"Penalty on the offense for an illegal forward pass, 5 yards",penalty,,,,-5,none,test,standard
t64,Marcus to #2 for 9,pass,Marcus,Riley,,9,complete,test,standard
t65,"#7 threw to #12, 21 yards",pass,Marcus,Devon,,21,complete,test,standard
t66,Sam threw to Jaylan for 13,pass,Sam,Jaylen,,13,complete,test,standard
t67,"Marcus hit Tasha for 6, touchdown",pass,Marcus,Tasha,,6,touchdown,test,standard
t68,Devon ran 3 yards,run,,,Devon,3,none,test,standard
t69,Marcus's pass to Devon was intercepted,pass,Marcus,Devon,,0,interception,test,standard
t70,Riley ran for 18 yards and a touchdown,run,,,Riley,18,touchdown,test,standard
t71,Tasha ran for a 6 yard loss,run,,,Tasha,-6,none,test,standard
t72,Sam hit Ri for 7,pass,Sam,Riley,,7,complete,test,standard
t73,Jay ran 9 yards,run,,,Jaylen,9,none,test,standard
t74,Sammy threw it away,pass,Sam,,,0,incomplete,test,standard
t75,Devon caught a pass from Sam for 10 yards,pass,Sam,Devon,,10,complete,test,standard
t76,"Dev dropped the ball from Marcus, incomplete",pass,Marcus,Devon,,0,incomplete,test,standard
t77,Marcus threw a 15 yard touchdown to Jaylen,pass,Marcus,Jaylen,,15,touchdown,test,standard
t78,Tasha took a handoff for 5,run,,,Tasha,5,none,test,standard
t79,Sam threw a pick intended for Jaylen,pass,Sam,Jaylen,,0,interception,test,standard
t80,Marcus ran for 2 yards,run,,,Marcus,2,none,test,standard
h01,The ball was thrown by Sam to Tasha for 9 yards,pass,Sam,Tasha,,9,complete,hard,passive voice
h02,"Jaylen was targeted by Marcus, but it fell incomplete",pass,Marcus,Jaylen,,0,incomplete,hard,passive voice
h03,"A 14 yard completion to Riley, thrown by Marcus",pass,Marcus,Riley,,14,complete,hard,passive voice
h04,Devon was stopped for a 3 yard loss,run,,,Devon,-3,none,hard,passive voice
h05,"The pass from Sam was intercepted, intended for Dev",pass,Sam,Devon,,0,interception,hard,passive voice
h06,Marcus completed one to Jaylen,pass,Marcus,Jaylen,,0,complete,hard,missing yards
h07,Riley rushed for a score,run,,,Riley,0,touchdown,hard,missing yards
h08,"Sam threw to Tasha, caught",pass,Sam,Tasha,,0,complete,hard,missing yards
h09,Penalty on the defense,penalty,,,,0,none,hard,missing yards
h10,Tasha ran,run,,,Tasha,0,none,hard,missing yards
h11,Sam threw to Jaylen at the 30 for 12 yards,pass,Sam,Jaylen,,12,complete,hard,extra numbers
h12,"On 3rd and 8, Marcus hit Tasha for 11",pass,Marcus,Tasha,,11,complete,hard,extra numbers
h13,"Down by 7, Devon ran for 6 yards in the 4th quarter",run,,,Devon,6,none,hard,extra numbers
h14,From the 15 yard line Riley ran 15 yards for a touchdown,run,,,Riley,15,touchdown,hard,extra numbers
h15,Marcous threw to Taysha for 8,pass,Marcus,Tasha,,8,complete,hard,misspelled names
h16,Sam threw to #21 for 9 yards,pass,Sam,,,9,complete,hard,player not on roster
h17,Chris ran for 5 yards,run,,,,5,none,hard,player not on roster
h18,Devin ran for 6,run,,,Devon,6,none,hard,misspelled names
h19,"#7 to #12, 17 yards",pass,Marcus,Devon,,17,complete,hard,jersey numbers
h20,"Number 15 handed off to number 4, who gained 8",run,,,Jaylen,8,none,hard,jersey numbers
h21,"Marcus launched a bomb to Jay, house call, 45 yards",pass,Marcus,Jaylen,,45,touchdown,hard,slang
h22,"Sam got picked, Dev was the target",pass,Sam,Devon,,0,interception,hard,slang
h23,Tasha juked two guys and took it 13,run,,,Tasha,13,none,hard,slang
h24,"Marcus airmailed it, nobody there",pass,Marcus,,,0,incomplete,hard,slang
h25,Timeout called by the offense,no_play,,,,0,none,hard,non-play
h26,End of the first half,no_play,,,,0,none,hard,non-play
h27,Marcus checked with the ref before the snap,no_play,,,,0,none,hard,non-play
h28,Two minute warning,no_play,,,,0,none,hard,non-play
h29,"Marcus threw incomplete to Tasha, then Sam ran for 5 on the next play",run,,,Sam,5,none,hard,two plays (label last)
h30,"Marcus connected with Jay for 6, then a flag on the offense, holding, 10 yards",penalty,,,,-10,none,hard,two plays (label last)
"""

In [ ]:
# 2. Roster, schema, data
ROSTER = {
    "Marcus": {"number": 7,  "aliases": ["marcus", "marc"]},
    "Jaylen": {"number": 4,  "aliases": ["jaylen", "jay", "j-money"]},
    "Devon":  {"number": 12, "aliases": ["devon", "dev"]},
    "Tasha":  {"number": 9,  "aliases": ["tasha", "tash"]},
    "Riley":  {"number": 2,  "aliases": ["riley", "ri"]},
    "Sam":    {"number": 15, "aliases": ["sam", "sammy"]},
}
FIELDS     = ["play_type", "passer", "receiver", "runner", "yards", "result"]
PLAY_TYPES = {"pass", "run", "penalty", "no_play"}
RESULTS    = {"complete", "incomplete", "interception", "touchdown", "none"}

def _n(x): return None if x in ("", None) else x
def load_rows():
    out = []
    for r in csv.DictReader(io.StringIO(DATA_CSV)):
        gold = dict(play_type=r["play_type"], passer=_n(r["passer"]), receiver=_n(r["receiver"]),
                    runner=_n(r["runner"]), yards=int(r["yards"]), result=r["result"])
        out.append(dict(id=r["id"], sentence=r["sentence"], split=r["split"], difficulty=r["difficulty"], gold=gold))
    return out
ALL  = load_rows()
DEV  = [r for r in ALL if r["split"] == "dev"]
EVAL = [r for r in ALL if r["split"] in ("test", "hard")]
print({s: sum(r["split"] == s for r in ALL) for s in ("dev", "test", "hard")}, "| evaluated on", len(EVAL))

## 3. Baseline: rule-based parser (regex + keywords). Frozen: not edited for this dataset, so it does not know the `no_play` class.

In [ ]:
RUN_RE  = re.compile(r"\b(ran|runs?|rush(ed|es)?|scrambl\w*|hand(ed)? ?off|carr(y|ied|ies)|took it)\b")
PASS_RE = re.compile(r"\b(threw|throws?|pass(es|ed)?|caught|catch|found|fired|tossed|completed|connected)\b")

def _find_players(text):
    t = text.lower(); hits = []
    for name, info in ROSTER.items():
        for alias in info["aliases"]:
            for m in re.finditer(r"(?<![\w-])" + re.escape(alias) + r"(?![\w-])", t):
                hits.append((m.start(), name))
        for m in re.finditer(r"#\s?" + str(info["number"]) + r"(?!\d)", t):
            hits.append((m.start(), name))
    return sorted(hits)

def _yards(t):
    if "no gain" in t: return 0
    m = (re.search(r"(\d+)\s*[- ]?\s*yard", t)
         or re.search(r"\bfor (?:a )?(?:gain of )?(\d+)\b", t)
         or re.search(r"\b(?:gained|gain of|got|went|picked up|loss of|lost)\s+(\d+)\b", t))
    return int(m.group(1)) if m else 0

def baseline_parse(text):
    t = text.lower()
    hits = _find_players(text)
    names = list(dict.fromkeys(n for _, n in hits))
    from_player = next((n for pos, n in hits if re.search(r"\bfrom\s+$", t[max(0, pos - 8):pos])), None)
    if re.search(r"penalt|flag on|holding|false start|interference|offside", t):
        play_type = "penalty"
    elif RUN_RE.search(t):
        play_type = "run"
    elif PASS_RE.search(t) or re.search(r"intercept|picked off|\bpick\b|incomplete", t):
        play_type = "pass"
    elif len(names) >= 2:
        play_type = "pass"
    else:
        play_type = "run"
    if re.search(r"intercept|picked off|\bpick\b", t):   result = "interception"
    elif re.search(r"\b(td|touchdown|scored)\b", t):     result = "touchdown"
    elif re.search(r"incomplete|dropped", t):            result = "incomplete"
    elif play_type == "pass":                            result = "complete"
    else:                                                result = "none"
    if play_type == "penalty": result = "none"
    yards = _yards(t)
    if play_type == "penalty":
        if re.search(r"holding|false start|\boffense\b|offensive", t) and "defens" not in t:
            yards = -yards
    elif re.search(r"\b(loss|lost|lose)\b", t):
        yards = -yards
    if result in ("incomplete", "interception"): yards = 0
    passer = receiver = runner = None
    if play_type == "pass":
        if from_player:
            passer = from_player
            rest = [n for n in names if n != from_player]
            receiver = rest[0] if rest else None
        else:
            passer = names[0] if names else None
            receiver = names[1] if len(names) > 1 else None
    elif play_type == "run" and names:
        runner = names[-1] if re.search(r"hand(ed)? ?off", t) else names[0]
    return dict(play_type=play_type, passer=passer, receiver=receiver, runner=runner, yards=yards, result=result)
print(baseline_parse("Marcus threw to Jaylen for 12 yards"))

## 4. Scoring (identical for every system)
Whole-entry accuracy = all 6 fields correct. Per-field accuracy = correct fields / (sentences x 6). A failed or malformed AI reply counts as fully wrong. 95% intervals are Wilson intervals; the paired test is an exact McNemar test on whole-entry correctness.

In [ ]:
def row_ok(pred, gold):
    if pred is None or "_error" in pred: return {f: False for f in FIELDS}
    return {f: pred.get(f) == gold[f] for f in FIELDS}

def score(preds, rows):
    oks = [row_ok(p, r["gold"]) for p, r in zip(preds, rows)]
    whole = [all(o.values()) for o in oks]
    fails = sum(1 for p in preds if p is None or "_error" in p)
    return dict(whole=whole, whole_n=sum(whole), n=len(rows), fails=fails,
                field={f: sum(o[f] for o in oks) for f in FIELDS})

def wilson(k, n, z=1.96):
    if n == 0: return (0, 0)
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d; h = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return (max(0, c-h), min(1, c+h))

def mcnemar_exact(a_ok, b_ok):
    b = sum((not x) and y for x, y in zip(a_ok, b_ok))   # A wrong, B right
    c = sum(x and (not y) for x, y in zip(a_ok, b_ok))   # A right, B wrong
    n = b + c
    if n == 0: return b, c, 1.0
    k = min(b, c)
    p = min(1.0, 2*sum(math.comb(n, i) for i in range(k+1)) / 2**n)
    return b, c, p

## 5. LLM parsers (zero-shot vs few-shot; Haiku vs Sonnet)
Both prompts share the same schema and labeling rules. The few-shot prompt only adds the 20 DEV examples. No temperature is set (API default), so each system is run `REPEATS` times to measure run-to-run variation.

In [ ]:
def _roster_lines():
    return "\n".join(f"- {n} (jersey #{i['number']}); nicknames: {', '.join(a for a in i['aliases'] if a != n.lower())}"
                     for n, i in ROSTER.items())

RULES = f'''You convert one plain-English description of a flag football play into a JSON record.

Roster (use ONLY these canonical first names in the output):
{_roster_lines()}

Return ONLY a JSON object with exactly these keys, and nothing else (no prose, no code fences):
  "play_type": "pass" | "run" | "penalty" | "no_play"
  "passer":    roster name or null
  "receiver":  roster name or null (null if the pass was thrown away or no target is named)
  "runner":    roster name or null (the ball carrier on a run, including a QB scramble)
  "yards":     integer, yards gained by the offense
  "result":    "complete" | "incomplete" | "interception" | "touchdown" | "none"

Rules:
- Resolve nicknames, typos, and jersey numbers to the canonical roster name. If a player is not on the roster, use null for that field.
- Pass: passer and receiver are set (receiver null if none named), runner is null. Run: runner is set, passer and receiver are null (a handoff giver is not recorded). Penalty: all three names are null.
- no_play: timeouts, period ends, or anything that is not a play. All names null, yards 0, result "none".
- yards is NEGATIVE for a loss of yards and for offensive penalties (holding, false start, delay of game). Defensive penalties are positive. No gain = 0. If no yardage is stated, use 0. Field position (e.g. "from the 30") and down/distance are not yards gained.
- Incomplete passes and interceptions have yards 0.
- result: "complete" for a caught pass, "touchdown" if a touchdown was scored (pass or run; keep the yards of the play), "none" for runs, penalties and no_play.
- If a receiver catches a short pass and then runs with it, it is still a pass; yards is the total gained.
- If a sentence describes two plays, record only the final play described.
- Never invent information that is not in the sentence.
'''
def _examples():
    return "\n".join(f"Input: {r['sentence']}\nOutput: {json.dumps(r['gold'])}" for r in DEV)
PROMPT_ZERO = RULES
PROMPT_FEW  = RULES + "\nExamples:\n" + _examples() + "\n"

def _extract_json(raw):
    m = re.search(r"\{.*\}", raw, flags=re.DOTALL)
    return json.loads(m.group(0) if m else raw)

def _validate(rec):
    if set(rec) != set(FIELDS): raise ValueError(f"wrong keys: {sorted(rec)}")
    if rec["play_type"] not in PLAY_TYPES or rec["result"] not in RESULTS: raise ValueError("bad play_type/result")
    for f in ("passer", "receiver", "runner"):
        if rec[f] in ("", "null", "None"): rec[f] = None
    rec["yards"] = int(rec["yards"]); return rec

def call_model(model, system, text):
    # Some models return a thinking block before the text block, so join only the text blocks.
    r = client.messages.create(model=model, max_tokens=1024, system=system,
                               messages=[{"role": "user", "content": text}])
    return "".join(b.text for b in r.content if getattr(b, "type", "") == "text")

def parse_with(model, system, text, tries=5):
    for a in range(tries):
        try:
            return _validate(_extract_json(call_model(model, system, text)))
        except Exception as e:
            code = getattr(e, "status_code", None)
            transient = code in (429, 500, 502, 503, 529) or type(e).__name__ in ("APIConnectionError", "APITimeoutError")
            if transient and a < tries - 1: time.sleep(2 ** a); continue
            raise

def run_system(model, system, rows):
    def one(r):
        try: return parse_with(model, system, r["sentence"])
        except Exception as e: return {"_error": f"{type(e).__name__}: {str(e)[:150]}"}
    with ThreadPoolExecutor(WORKERS) as ex: return list(ex.map(one, rows))

## 6. API key check
Paste the key string (starts with `sk-ant-`), not the key's name. Nothing is saved.

In [ ]:
client = None
try:
    import anthropic
    key = (os.environ.get("ANTHROPIC_API_KEY") or getpass.getpass("Paste Anthropic API key: ")).strip().strip('"').strip("'")
    print("Starts:", key[:11], "| Ends:", key[-4:], "| length:", len(key), "| anthropic", anthropic.__version__)
    client = anthropic.Anthropic(api_key=key)
    client.messages.create(model=HAIKU, max_tokens=10, messages=[{"role": "user", "content": "hi"}])
    print("KEY VERIFIED.")
except Exception as e:
    client = None
    print("KEY CHECK FAILED:", type(e).__name__, str(e)[:200])

SONNET_OK = False
if client is not None:
    try:
        client.messages.create(model=SONNET, max_tokens=10, messages=[{"role": "user", "content": "hi"}])
        SONNET_OK = True; print("Sonnet available.")
    except Exception as e:
        print("Sonnet NOT available (", type(e).__name__, str(e)[:120], ") - Sonnet rows will be marked not run.")

## 7. Run all systems on the same 110 held-out sentences

In [ ]:
SYSTEMS = [("Rule-based baseline", None, None),
           ("Haiku 4.5 zero-shot", HAIKU, PROMPT_ZERO),
           ("Haiku 4.5 few-shot", HAIKU, PROMPT_FEW),
           ("Sonnet 5.5 few-shot", SONNET, PROMPT_FEW)]
RUNS = {}     # system -> list of prediction lists (one per repeat)
if client is None:
    print("No working client: LLM systems NOT RUN. Fix the key cell and rerun from there.")
for name, model, system in SYSTEMS:
    if model is None:
        RUNS[name] = [[baseline_parse(r["sentence"]) for r in EVAL]]; continue
    if client is None or (model == SONNET and not SONNET_OK): continue
    RUNS[name] = []
    for rep in range(REPEATS):
        t0 = time.time(); preds = run_system(model, system, EVAL); RUNS[name].append(preds)
        s = score(preds, EVAL)
        print(f"{name} run {rep+1}: whole {s['whole_n']}/{s['n']} | parse failures {s['fails']} | {time.time()-t0:.0f}s")

## 8. Results table

In [ ]:
SPLITS = {"test": [i for i, r in enumerate(EVAL) if r["split"] == "test"],
          "hard": [i for i, r in enumerate(EVAL) if r["split"] == "hard"],
          "all":  list(range(len(EVAL)))}
def sub(preds, idx): return [preds[i] for i in idx]
def sub_rows(idx):   return [EVAL[i] for i in idx]

table, RES = [], {}
for name, _, _ in SYSTEMS:
    runs = RUNS.get(name)
    if not runs:
        table.append({"System": name, **{f"{s} whole-entry": "NOT RUN" for s in SPLITS}, "all per-field": "NOT RUN", "parse failures": "-", "runs": 0}); continue
    row = {"System": name}; RES[name] = {}
    for s, idx in SPLITS.items():
        sc = [score(sub(p, idx), sub_rows(idx)) for p in runs]
        accs = [x["whole_n"] / x["n"] for x in sc]; mean = sum(accs) / len(accs); n = sc[0]["n"]
        lo, hi = wilson(mean * n, n)
        row[f"{s} whole-entry"] = f"{mean:.1%} [{lo:.0%}-{hi:.0%}]" + (f" (runs {min(accs):.0%}-{max(accs):.0%})" if len(accs) > 1 else "")
        RES[name][s] = dict(n=n, mean=mean, lo=lo, hi=hi, runs=accs, fields={f: sum(x["field"][f] for x in sc) / len(sc) / n for f in FIELDS})
        if s == "all":
            row["all per-field"] = f"{sum(sum(x['field'].values()) for x in sc) / len(sc) / (n*6):.1%}"
            row["parse failures"] = "/".join(str(x["fails"]) for x in sc)
    row["runs"] = len(runs); table.append(row)
TAB = pd.DataFrame(table); pd.set_option("display.width", 250, "display.max_colwidth", 80)
display(TAB); TAB.to_csv("results_table.csv", index=False)

print("\nPaired exact McNemar tests vs baseline (run 1, all 110 sentences):")
base_ok = score(RUNS["Rule-based baseline"][0], EVAL)["whole"]
PAIRED = {}
for name in RES:
    if name == "Rule-based baseline": continue
    b, c, p = mcnemar_exact(base_ok, score(RUNS[name][0], EVAL)["whole"])
    PAIRED[name] = dict(baseline_wrong_system_right=b, baseline_right_system_wrong=c, p=p)
    print(f"  {name}: system fixes {b}, system breaks {c}, p = {p:.4g}")
if "Haiku 4.5 zero-shot" in RES and "Haiku 4.5 few-shot" in RES:
    b, c, p = mcnemar_exact(score(RUNS["Haiku 4.5 zero-shot"][0], EVAL)["whole"], score(RUNS["Haiku 4.5 few-shot"][0], EVAL)["whole"])
    PAIRED["few vs zero (Haiku)"] = dict(zero_wrong_few_right=b, zero_right_few_wrong=c, p=p)
    print(f"  Haiku few-shot vs zero-shot: few fixes {b}, few breaks {c}, p = {p:.4g}")

## 9. Per-field and per-category results

In [ ]:
print("Per-field accuracy on all 110 (mean over runs):")
display(pd.DataFrame({n: {f: f"{RES[n]['all']['fields'][f]:.1%}" for f in FIELDS} for n in RES}))

hard_idx = SPLITS["hard"]; cats = sorted({EVAL[i]["difficulty"] for i in hard_idx})
cat_rows = {}
for name in RES:
    cat_rows[name] = {}
    for c in cats:
        idx = [i for i in hard_idx if EVAL[i]["difficulty"] == c]
        accs = [score(sub(p, idx), sub_rows(idx))["whole_n"] / len(idx) for p in RUNS[name]]
        cat_rows[name][f"{c} (n={len(idx)})"] = sum(accs) / len(accs)
CAT = pd.DataFrame(cat_rows); display((CAT * 100).round(0).astype(int))

## 10. Figures

In [ ]:
names = list(RES)
fig, ax = plt.subplots(figsize=(8, 4)); w = 0.38
for j, (s, lab) in enumerate([("test", "Test (n=80)"), ("hard", "Hard (n=30)")]):
    ys = [RES[n][s]["mean"] * 100 for n in names]
    err = [[(RES[n][s]["mean"] - RES[n][s]["lo"]) * 100 for n in names], [(RES[n][s]["hi"] - RES[n][s]["mean"]) * 100 for n in names]]
    bars = ax.bar([i + (j - 0.5) * w for i in range(len(names))], ys, w, yerr=err, capsize=3, label=lab)
    ax.bar_label(bars, fmt="%.0f%%", padding=2, fontsize=8)
ax.set_xticks(range(len(names))); ax.set_xticklabels([n.replace(" ", "\n", 1) for n in names], fontsize=8)
ax.set_ylabel("Whole-entry accuracy (%)"); ax.set_ylim(0, 115); ax.legend(loc="lower right")
ax.set_title("Whole-entry accuracy by system and split (bars: 95% Wilson interval)")
plt.tight_layout(); plt.savefig("fig_main.png", dpi=170); plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5)); im = ax.imshow(CAT.values * 100, vmin=0, vmax=100, cmap="RdYlGn", aspect="auto")
ax.set_xticks(range(len(CAT.columns))); ax.set_xticklabels([c.replace(" ", "\n", 1) for c in CAT.columns], fontsize=8)
ax.set_yticks(range(len(CAT.index))); ax.set_yticklabels(CAT.index, fontsize=8)
for i in range(CAT.shape[0]):
    for j in range(CAT.shape[1]): ax.text(j, i, f"{CAT.values[i, j]*100:.0f}", ha="center", va="center", fontsize=8)
ax.set_xlabel("System"); ax.set_ylabel("Hard-set category"); ax.set_title("Whole-entry accuracy (%) by hard-set category")
plt.colorbar(im, label="%"); plt.tight_layout(); plt.savefig("fig_category.png", dpi=170); plt.show()

## 11. Error analysis (run 1) and saved outputs

In [ ]:
err_rows = []
for name in RUNS:
    for r, p in zip(EVAL, RUNS[name][0]):
        if p is None or "_error" in p:
            err_rows.append(dict(system=name, id=r["id"], split=r["split"], difficulty=r["difficulty"], sentence=r["sentence"], field="PARSE FAILURE", predicted=(p or {}).get("_error"), gold=""))
            continue
        for f in FIELDS:
            if p.get(f) != r["gold"][f]:
                err_rows.append(dict(system=name, id=r["id"], split=r["split"], difficulty=r["difficulty"], sentence=r["sentence"], field=f, predicted=p.get(f), gold=r["gold"][f]))
ERR = pd.DataFrame(err_rows); ERR.to_csv("errors.csv", index=False)
print("Wrong fields per system (run 1):"); display(ERR.groupby("system").size() if len(ERR) else "none")
for name in RUNS:
    if name == "Rule-based baseline": continue
    e = ERR[ERR.system == name]
    print(f"\n=== {name}: {e.id.nunique()} sentences with an error ===")
    for sid, g in e.groupby("id"):
        print(f'- [{sid}] "{g.sentence.iloc[0]}"')
        for _, x in g.iterrows(): print(f"    {x.field}: predicted {x.predicted!r}, correct {x.gold!r}")

# run-to-run consistency
for name in RUNS:
    if len(RUNS[name]) > 1:
        same = sum(json.dumps(a, sort_keys=True) == json.dumps(b, sort_keys=True) for a, b in zip(RUNS[name][0], RUNS[name][1]))
        print(f"{name}: identical predictions across run 1 and 2 on {same}/{len(EVAL)} sentences")

json.dump(dict(models=dict(haiku=HAIKU, sonnet=SONNET), repeats=REPEATS, n_eval=len(EVAL),
               results=RES, paired_tests=PAIRED, per_category=CAT.to_dict(), date=str(datetime.date.today())),
          open("results.json", "w"), indent=2, default=str)
json.dump({n: RUNS[n] for n in RUNS}, open("predictions.json", "w"), indent=1, default=str)
print("\nsaved results.json, results_table.csv, predictions.json, errors.csv, fig_main.png, fig_category.png")

## 12. Demo: type a sentence, see the structured records

In [ ]:
def demo(s):
    print("INPUT   :", s); print("BASELINE:", baseline_parse(s))
    if client is not None:
        try: print("CLAUDE  :", parse_with(HAIKU, PROMPT_FEW, s))
        except Exception as e: print("CLAUDE  : failed ->", e)
    print()
demo("Marcus dumped it off to Jay, who took it 8 yards")
demo("Sam threw to Tasha, she ran it 20 yards for a score")
demo("The ball was thrown by Sam to Tasha for 9 yards")
# demo("type your own sentence here")

## Reproducibility notes
- Hardware: Google Colab CPU runtime (no GPU needed; models are called through the Anthropic API).
- Models: `claude-haiku-4-5-20251001`, `claude-sonnet-5-5`. No temperature set (API default); `max_tokens=1024`; each LLM system run twice.
- Splits: dev (20) used only as prompt examples; test (80) and hard (30): the prompt and labeling conventions were finalized before any model was run on these sentences and were not edited afterward (but the same author and AI assistant wrote both the conventions and the sentences). The baseline was written for an earlier course checkpoint and was not edited for this dataset. After a first run, the harness was fixed to read only text blocks (Sonnet can return a thinking block first) and max_tokens was raised to 1024 for all systems; everything was then re-run.
- The API key is entered at run time and is not stored in this notebook.